In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
file_path = "../dataset/prepared/ClassifierPreparedDataset.csv"
df = pd.read_csv(file_path)

# Optional: Preview the first few rows
df.head()


In [ ]:
X = df.drop(columns=["NLOS"])
y = df["NLOS"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=111)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create and train the Logistic Regression model
logreg = LogisticRegression(random_state=111, max_iter=500)
logreg.fit(X_train, y_train)

# 2. Predict on the test set
y_pred_logreg = logreg.predict(X_test)

# 3. Evaluate the performance
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
print("Logistic Regression Test Accuracy: {:.4f}".format(accuracy_logreg))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logreg))

# 4. Compute and display the confusion matrix
cm_logreg = confusion_matrix(y_test, y_pred_logreg)
print("Confusion Matrix:")
print(cm_logreg)

plt.figure(figsize=(6,4))
sns.heatmap(cm_logreg, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NLOS=0", "NLOS=1"],
            yticklabels=["NLOS=0", "NLOS=1"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your training features are in X_train (with the target dropped)
# If not, create a new DataFrame for feature importance analysis:
# X_train_no_target = X_train.drop(columns=["NLOS"])
# Otherwise, if X_train is already defined without "NLOS":
X_train_no_target = X_train

# Extract coefficients from the logistic regression model.
# For binary classification, logreg.coef_ returns a 2D array of shape (1, n_features)
coefficients = pd.Series(logreg.coef_[0], index=X_train_no_target.columns)

# Sort coefficients by absolute value (largest magnitude first)
coefficients_sorted = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index)

plt.figure(figsize=(8,5))
ax = sns.barplot(x=coefficients_sorted.values, 
                 y=coefficients_sorted.index, 
                 hue=coefficients_sorted.index, 
                 palette="viridis", 
                 dodge=False)
if ax.get_legend() is not None:
    ax.get_legend().remove()

plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.title("Logistic Regression Coefficients (Feature Importance)")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# -----------------------------
# Remove Outliers (Noise) Detected by DBSCAN
# -----------------------------

# Add DBSCAN cluster labels to the dataframe
df["cluster"] = clusters

# Separate out the noise (-1) from valid data
df_filtered = df[df["cluster"] != -1].drop(columns=["cluster"])  # Keep only non-noise points

# Re-define X and y after outlier removal
X_filtered = df_filtered.drop(columns=["NLOS"])
y_filtered = df_filtered["NLOS"]

# Rescale the filtered dataset
scaler = StandardScaler()
X_filtered_scaled = scaler.fit_transform(X_filtered)

# -----------------------------
# Train Logistic Regression Again After Noise Removal
# -----------------------------

# Split dataset after noise removal
X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(
    X_filtered, y_filtered, test_size=0.3, random_state=111)

# Train Logistic Regression (After Outlier Removal)
logreg_after = LogisticRegression(random_state=111, max_iter=500)
logreg_after.fit(X_train_filtered, y_train_filtered)
y_pred_after = logreg_after.predict(X_test_filtered)
accuracy_after = accuracy_score(y_test_filtered, y_pred_after)

# -----------------------------
# Compare Accuracy Before and After Outlier Removal
# -----------------------------
accuracy_results = pd.DataFrame({
    "Condition": ["Before Outlier Removal", "After Outlier Removal"],
    "Accuracy": [round(accuracy_logreg, 4), round(accuracy_after, 4)]
})

print("\nAccuracy Comparison:")
print(accuracy_results)

# -----------------------------
# Visualize DBSCAN Clusters Before and After Noise Removal
# -----------------------------

# PCA Projection for Visualization
pca = PCA(n_components=2)
X_pca_before = pca.fit_transform(X_scaled)  # Before noise removal
X_pca_after = pca.fit_transform(X_filtered_scaled)  # After noise removal

# Create DataFrames for visualization
pca_df_before = pd.DataFrame(X_pca_before, columns=["PCA1", "PCA2"])
pca_df_before["cluster"] = clusters

pca_df_after = pd.DataFrame(X_pca_after, columns=["PCA1", "PCA2"])
pca_df_after["cluster"] = DBSCAN(eps=eps_value, min_samples=min_samples).fit_predict(X_filtered_scaled)

# Plot Before Noise Removal
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df_before, x="PCA1", y="PCA2", hue="cluster", palette="viridis", s=60)
plt.title("DBSCAN Clusters Before Noise Removal")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

# Plot After Noise Removal
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df_after, x="PCA1", y="PCA2", hue="cluster", palette="viridis", s=60)
plt.title("DBSCAN Clusters After Noise Removal")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# -----------------------------
# Apply K-Means for Clustering (Instead of DBSCAN)
# -----------------------------

kmeans = KMeans(n_clusters=3, random_state=111, n_init=10)  # Adjust the number of clusters as needed
df["cluster"] = kmeans.fit_predict(X_scaled)  # Assign clusters

# Find the smallest cluster (assume it contains outliers)
cluster_counts = df["cluster"].value_counts()
smallest_cluster = cluster_counts.idxmin()  # Cluster with the fewest data points

# Remove outliers: Keep only points that are **not** in the smallest cluster
df_filtered = df[df["cluster"] != smallest_cluster].drop(columns=["cluster"])  # Keep valid points

# Re-define X and y after outlier removal
X_filtered = df_filtered.drop(columns=["NLOS"])
y_filtered = df_filtered["NLOS"]

# Rescale the filtered dataset
scaler = StandardScaler()
X_filtered_scaled = scaler.fit_transform(X_filtered)

# -----------------------------
# Train Logistic Regression Again After Noise Removal
# -----------------------------

# Split dataset after noise removal
X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(
    X_filtered, y_filtered, test_size=0.3, random_state=111)

# Train Logistic Regression (After Outlier Removal with K-Means)
logreg_after = LogisticRegression(random_state=111, max_iter=500)
logreg_after.fit(X_train_filtered, y_train_filtered)
y_pred_after = logreg_after.predict(X_test_filtered)
accuracy_after = accuracy_score(y_test_filtered, y_pred_after)

# -----------------------------
# Compare Accuracy Before and After Outlier Removal
# -----------------------------
accuracy_results = pd.DataFrame({
    "Condition": ["Before Outlier Removal", "After Outlier Removal (K-Means)"],
    "Accuracy": [round(accuracy_logreg, 4), round(accuracy_after, 4)]
})

print("\nAccuracy Comparison with K-Means:")
print(accuracy_results.to_string(index=False))  # Print without row index

# -----------------------------
# Visualize K-Means Clusters Before and After Outlier Removal
# -----------------------------

# PCA Projection for Visualization
pca = PCA(n_components=2)
X_pca_before = pca.fit_transform(X_scaled)  # Before noise removal
X_pca_after = pca.fit_transform(X_filtered_scaled)  # After noise removal

# Create DataFrames for visualization
pca_df_before = pd.DataFrame(X_pca_before, columns=["PCA1", "PCA2"])
pca_df_before["cluster"] = kmeans.labels_

pca_df_after = pd.DataFrame(X_pca_after, columns=["PCA1", "PCA2"])
pca_df_after["cluster"] = KMeans(n_clusters=3, random_state=111, n_init=10).fit_predict(X_filtered_scaled)

# Plot Before Outlier Removal
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df_before, x="PCA1", y="PCA2", hue="cluster", palette="viridis", s=60)
plt.title("K-Means Clusters Before Noise Removal")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

# Plot After Outlier Removal
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df_after, x="PCA1", y="PCA2", hue="cluster", palette="viridis", s=60)
plt.title("K-Means Clusters After Noise Removal")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()
